# LDaCA App: Start Services and Open Frontend
This notebook starts the backend (FastAPI) and frontend (nginx/static proxy) from the notebook session, verifies health, and opens the app through Jupyter’s proxy.


In [ ]:
# Install minimal deps (requests)
import sys, subprocess, shlex, os, time, socket, atexit, signal, pathlib
from urllib.request import urlopen, Request
from urllib.error import URLError, HTTPError

try:
    import requests  # noqa: F401
except Exception:
    # Best-effort install if not present
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-input", "--quiet", "requests"], check=False)

from IPython.display import HTML, display

ROOT = pathlib.Path.cwd()
HOME = pathlib.Path(os.environ.get("HOME", str(ROOT)))
LOG_DIR = HOME / "ldaca_nb_logs"
LOG_DIR.mkdir(exist_ok=True)

backend_port = int(os.environ.get("BACKEND_PORT", "8001"))
frontend_port = int(os.environ.get("FRONTEND_PORT", "8080"))

backend_cmd = os.environ.get("BACKEND_CMD") or f"uvicorn main:app --host 127.0.0.1 --port {backend_port} --log-level info"
backend_cwd = ROOT / "ldaca_web_app" / "backend"

# For frontend we rely on nginx serving prebuilt assets via start script; here we ensure nginx is running using the start script if available.
frontend_cmd = os.environ.get("FRONTEND_CMD") or f"bash start"
frontend_cwd = ROOT

procs = {}

def wait_port(host: str, port: int, timeout: float = 30.0):
    end = time.time() + timeout
    while time.time() < end:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(1)
            try:
                s.connect((host, port))
                return True
            except Exception:
                time.sleep(0.5)
    return False

def start_process(key: str, cmd: str, cwd: pathlib.Path, log_name: str):
    logfile = LOG_DIR / log_name
    f = open(logfile, "ab", buffering=0)
    # Write a header
    f.write(f"\n===== Starting {key}: {cmd} (cwd={cwd}) =====\n".encode())
    proc = subprocess.Popen(shlex.split(cmd), cwd=str(cwd), stdout=f, stderr=subprocess.STDOUT, env=os.environ.copy())
    procs[key] = (proc, f)
    return proc, logfile

# Ensure PYTHONPATH includes repo paths so backend imports resolve
extra_paths = [str(ROOT), str(ROOT/"docframe"), str(ROOT/"docworkspace"), str(ROOT/"ldaca_web_app"/"backend")]
pp = os.environ.get("PYTHONPATH", "")
for p in extra_paths:
    if p and p not in pp:
        pp = (pp + (":" if pp else "")) + p
os.environ["PYTHONPATH"] = pp

# Start backend first if not already listening
backend_ready = wait_port("127.0.0.1", backend_port, timeout=1)
if not backend_ready:
    start_process("backend", backend_cmd, backend_cwd, "backend.log")
    backend_ready = wait_port("127.0.0.1", backend_port, timeout=60)

# Start frontend/nginx via start script if not already listening
frontend_ready = wait_port("127.0.0.1", frontend_port, timeout=1)
if not frontend_ready:
    # Kill any stale nginx pid file to avoid startup errors
    try:
        pidfile = HOME/"nginx"/"run"/"nginx.pid"
        if pidfile.exists():
            pidfile.unlink()
    except Exception:
        pass
    start_process("frontend", frontend_cmd, frontend_cwd, "frontend.log")
    frontend_ready = wait_port("127.0.0.1", frontend_port, timeout=60)

# Poll health endpoints
backend_health_url = f"http://127.0.0.1:{backend_port}/health"
frontend_health_url = f"http://127.0.0.1:{frontend_port}/health"

def get_json(url):
    try:
        with urlopen(Request(url, headers={"Accept":"application/json"}), timeout=5) as r:
            return r.read().decode("utf-8"), True
    except Exception as e:
        return str(e), False

bh_content, bh_ok = get_json(backend_health_url)
fh_content, fh_ok = get_json(frontend_health_url)

# Tail logs
def tail(path: pathlib.Path, n: int = 80):
    if not path.exists():
        return "<no log>"
    try:
        with open(path, "rb") as f:
            return b"".join(f.readlines()[-n:]).decode(errors="replace")
    except Exception as e:
        return f"<error reading log: {e}>"

backend_log = LOG_DIR/"backend.log"
frontend_log = LOG_DIR/"frontend.log"

status_html = f"""
<div style='font-family: system-ui, sans-serif; line-height:1.4'>
  <h3>Service Status</h3>
  <ul>
    <li>Backend (port {backend_port}): <b style='color:{'green' if backend_ready else 'red'}'>{'READY' if backend_ready else 'NOT READY'}</b> — <a href="/proxy/{backend_port}/api/docs" target="_blank">/proxy/{backend_port}/api/docs</a></li>
    <li>Frontend (port {frontend_port}): <b style='color:{'green' if frontend_ready else 'red'}'>{'READY' if frontend_ready else 'NOT READY'}</b> — <a href="/proxy/{frontend_port}/" target="_blank">/proxy/{frontend_port}/</a></li>
  </ul>
  <details open><summary>Backend health {"✅" if bh_ok else "❌"}</summary>
    <pre style='white-space:pre-wrap; max-height:200px; overflow:auto'>{bh_content}</pre>
  </details>
  <details><summary>Frontend health {"✅" if fh_ok else "❌"}</summary>
    <pre style='white-space:pre-wrap; max-height:200px; overflow:auto'>{fh_content}</pre>
  </details>
  <details><summary>Backend log tail</summary>
    <pre style='white-space:pre-wrap; max-height:240px; overflow:auto'>{tail(backend_log)}</pre>
  </details>
  <details><summary>Frontend log tail</summary>
    <pre style='white-space:pre-wrap; max-height:240px; overflow:auto'>{tail(frontend_log)}</pre>
  </details>
</div>
"""

display(HTML(status_html))


In [ ]:
from IPython.display import Javascript, display, Markdown

url = f"/proxy/{8080}/"
# Attempt to open in new tab (may be blocked by popup settings)
display(Javascript(f"window.open('{url}', '_blank');"))
print("Opened new tab (if not blocked). Fallback link below:")
display(Markdown(f"[Open Frontend]({url})"))


In [ ]:
import atexit, signal, time
from ipywidgets import Button
from IPython.display import display

# Access the procs dict from Cell 1 via globals(), if present
_procs = globals().get('procs', {})


def _terminate_proc(name, proc_tuple, timeout=5):
    try:
        proc, f = proc_tuple
    except Exception:
        return
    if proc.poll() is None:
        try:
            proc.terminate()
        except Exception:
            pass
        t0 = time.time()
        while proc.poll() is None and (time.time() - t0) < timeout:
            time.sleep(0.2)
        if proc.poll() is None:
            try:
                proc.kill()
            except Exception:
                pass
    try:
        f.close()
    except Exception:
        pass


def _cleanup():
    for name, tup in list(_procs.items()):
        _terminate_proc(name, tup)

atexit.register(_cleanup)

btn = Button(description='Stop backend/frontend', button_style='warning')

def _on_click(_):
    _cleanup()
    print('Services stopped.')

btn.on_click(_on_click)
display(btn)
